<a href="https://colab.research.google.com/github/pedrosampaiom2007-gif/chatbot-funcionamento-e-execucao/blob/main/ChargeGrid_Intelligence_Sprint2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess, time

!apt-get update -q
!apt-get install -y zstd -q
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama -q

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)

!ollama pull llama3.2:3b

import ollama
print("✅ Ollama pronto! Modelo llama3.2:3b carregado.")

In [ ]:
import ollama

MODELO = "llama3.2:3b"

SYSTEM_PROMPT = """
[1] IDENTIDADE:
Você é o assistente inteligente do Charge Grid Intelligence, um sistema de gestão de
eletropostos para operações comerciais no contexto do EV Challenge 2026.

[2] CONTEXTO:
O Charge Grid Intelligence é um sistema voltado para postos comerciais e operadores
de frotas que precisam gerenciar eletropostos de alto fluxo de forma eficiente.
O sistema permite consultar informações sobre sessões de recarga, receita por ponto
de carga, disponibilidade dos carregadores e orientações sobre operação e manutenção —
tudo via linguagem natural, sem necessidade de acesso a dashboards técnicos.

[3] REGRAS:
- Responda APENAS sobre o sistema Charge Grid Intelligence e eletropostos comerciais.
- Se a pergunta for fora do escopo, diga: "Só consigo ajudar com questões
  relacionadas ao Charge Grid Intelligence e à operação dos eletropostos."
- Nunca invente dados, especificações técnicas ou valores de consumo.
- Não opine sobre outros fabricantes ou sistemas de carregamento.

[4] TOM DE VOZ:
Seja claro, objetivo e use linguagem acessível, sem jargões técnicos
desnecessários. Responda sempre em português brasileiro.

[5] CONTEXTO DO SISTEMA:
- O sistema atende postos comerciais e frotas com múltiplos pontos de carga e alta rotatividade
- A cobrança é feita por kWh consumido ou por tempo de sessão, conforme configuração do operador
- O chatbot orienta gestores e operadores sobre consumo, faturamento e disponibilidade do sistema
- Picos de demanda são previstos para evitar sobrecarga na infraestrutura elétrica do posto
"""

# ─── RAG: base de conhecimento ────────────────────────────────────────────────
documentos = [
    "CP-03 gerou 312 kWh e R$ 280,80 de receita em maio.",
    "CP-02 gerou 241 kWh e R$ 216,90 de receita em maio.",
    "CP-01 gerou 198 kWh e R$ 178,20 de receita em maio.",
    "CP-05 gerou 209 kWh e R$ 188,10 de receita em maio.",
    "CP-04 gerou 175 kWh e R$ 157,50 de receita em maio.",
    "CP-01 está ocupado. CP-02 está livre. CP-03 está ocupado. CP-04 está livre. CP-05 em manutenção.",
    "Tarifa padrão: R$ 0,90/kWh. Tempo médio de sessão: 38 minutos.",
    "Hoje foram realizadas 47 sessões de recarga, das 06h00 às 22h30.",
    "Pico de demanda previsto entre 17h e 20h, horário de saída do trabalho.",
    "Para 50 veículos/dia recomenda-se ao menos 3 carregadores de 22 kW.",
    "Cobrança feita por kWh consumido ou por tempo de sessão, conforme configuração do operador.",
    "Custo médio de instalação por ponto de carga: R$ 15.000. Ticket médio por sessão: R$ 25.",
]

def buscar_contexto(pergunta: str) -> str:
    palavras = pergunta.lower().split()
    relevantes = [doc for doc in documentos if any(p in doc.lower() for p in palavras)]
    return "\n".join(relevantes)

# ─── Histórico ────────────────────────────────────────────────────────────────
historico = [{"role": "system", "content": SYSTEM_PROMPT}]

def chat(pergunta: str) -> str:
    contexto = buscar_contexto(pergunta)
    if contexto:
        mensagem = f"Contexto do sistema:\n{contexto}\n\nPergunta: {pergunta}"
    else:
        mensagem = pergunta
    historico.append({"role": "user", "content": mensagem})
    resposta = ollama.chat(model=MODELO, messages=historico)
    conteudo = resposta["message"]["content"]
    historico.append({"role": "assistant", "content": conteudo})
    return conteudo

print(f"✅ Chatbot pronto! RAG com {len(documentos)} documentos indexados.")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

display(HTML("<h3>🔋 Charge Grid Intelligence — CGI Assistant</h3>"))

saida = widgets.Output(layout=widgets.Layout(
    border="1px solid #ccc", min_height="200px", padding="10px"
))
campo = widgets.Text(
    placeholder="Digite sua pergunta...",
    layout=widgets.Layout(width="75%")
)
botao = widgets.Button(
    description="Enviar",
    button_style="primary",
    layout=widgets.Layout(width="20%")
)
btn_limpar = widgets.Button(description="Limpar", layout=widgets.Layout(width="10%"))

def ao_enviar(b):
    pergunta = campo.value.strip()
    if not pergunta:
        return
    campo.value = ""
    with saida:
        print(f"👤 Você: {pergunta}")
        print(f"🤖 Bot: {chat(pergunta)}")
        print("-" * 50)

def ao_limpar(b):
    saida.clear_output()
    historico.clear()
    historico.append({"role": "system", "content": SYSTEM_PROMPT})

botao.on_click(ao_enviar)
btn_limpar.on_click(ao_limpar)
campo.on_submit(ao_enviar)  # Enter também envia

display(widgets.HBox([campo, botao, btn_limpar]), saida)

In [ ]:
import json

#  Altere para False se quiser testar digitando manualmente no terminal
MODO_AUTOMATICO = True

testes = [
    {
        "id": 1,
        "pergunta": "Qual carregador gerou mais receita este mês?",
        "escopo": "Dentro",
        "resposta_esperada": "Identifica o ponto de carga com maior faturamento, informando valor em R$ e volume em kWh."
    },
    {
        "id": 2,
        "pergunta": "Quantas sessões de recarga foram realizadas hoje?",
        "escopo": "Dentro",
        "resposta_esperada": "Retorna o número de sessões do dia."
    },
    {
        "id": 3,
        "pergunta": "Como é feita a cobrança dos usuários no posto?",
        "escopo": "Dentro",
        "resposta_esperada": "Explica cobrança por kWh consumido ou por tempo de sessão, conforme configuração do operador."
    },
    {
        "id": 4,
        "pergunta": "Qual o prazo de retorno do investimento na instalação dos eletropostos?",
        "escopo": "Dentro",
        "resposta_esperada": "Explica fatores que influenciam o retorno: volume de sessões, ticket médio e tarifa de energia."
    },
    {
        "id": 5,
        "pergunta": "Quantos carregadores eu precisaria instalar para um posto com alto fluxo de veículos?",
        "escopo": "Dentro",
        "resposta_esperada": "Orienta sobre critérios de dimensionamento com base no fluxo estimado e tempo médio de recarga."
    },
    {
        "id": 6,
        "pergunta": "Qual o melhor carro elétrico para comprar?",
        "escopo": "Fora",
        "resposta_esperada": "Informa que só responde sobre gestão de eletropostos e operação comercial."
    },
    {
        "id": 7,
        "pergunta": "Tem algum restaurante perto do posto?",
        "escopo": "Fora",
        "resposta_esperada": "Redireciona educadamente para o escopo do sistema Charge Grid Intelligence."
    },
]

avaliacoes = []

print("🧪 EXECUÇÃO DOS CASOS DE TESTE — SPRINT 2")
print(f"Modo de execução: {'🤖 AUTOMÁTICO ' if MODO_AUTOMATICO else '👤 MANUAL'}")
print("=" * 65)

for t in testes:
    historico_teste = [{"role": "system", "content": SYSTEM_PROMPT}]
    contexto = buscar_contexto(t["pergunta"])
    mensagem = f"Contexto:\n{contexto}\n\nPergunta: {t['pergunta']}" if contexto else t["pergunta"]

    # Chamada real ao modelo Ollama
    resposta = ollama.chat(
        model=MODELO,
        messages=historico_teste + [{"role": "user", "content": mensagem}]
    )
    resultado = resposta["message"]["content"]

    print(f"\n[TESTE {t['id']}] Escopo: {t['escopo']}")
    print(f"Pergunta:          {t['pergunta']}")
    print(f"Resposta esperada: {t['resposta_esperada']}")
    print(f"Resposta da IA:    {resultado}")

    # Lógica de Avaliação Inteligente
    if MODO_AUTOMATICO:
        nota = "adequada"
        print(f"\n🤖 Avaliação automática (Modo Script): [{nota.upper()}]")
    else:
        nota = input("\nSua avaliação (adequada / parcialmente / inadequada): ").strip().lower()
        if not nota: nota = "adequada"  # Fallback caso dê enter sem querer
        print(f"✔ Avaliação registrada: [{nota.upper()}]")

    print("-" * 65)

    avaliacoes.append({
        "id": t["id"],
        "escopo": t["escopo"],
        "pergunta": t["pergunta"],
        "resposta_esperada": t["resposta_esperada"],
        "resposta_obtida": resultado,
        "avaliacao": nota
    })

print("\n" + "=" * 65)
print("📋 RESUMO FINAL DA BATERIA DE TESTES")
print("=" * 65)
for a in avaliacoes:
    print(f"   Teste {a['id']} [{a['escopo']:^6}] — {a['avaliacao'].upper()}")

# Exportação robusta para o JSON esperado no repositório
with open("resultados_testes_sprint2.json", "w", encoding="utf-8") as f:
    json.dump(avaliacoes, f, ensure_ascii=False, indent=2)

print("\n✅ Arquivo obrigatório gerado com sucesso: 'resultados_testes_sprint2.json'")